In [ ]:
!pip install -q  accelerate bitsandbytes sentencepiece

In [ ]:
!pip install -U transformers

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/gemma-4-E2B-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
# 3. Prompt Hazırlığı
prompt = "Bana kuantum fiziğini, sanki 5 yaşında bir çocuğa masal anlatıyormuşsun gibi ama bir korsan ağzıyla anlat."

# Gemma-4 formatına uygun mesaj yapısı
chat = [
    {"role": "user", "content": prompt},
]
print(1)
formatted_prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

# 4. Çıkarım (Inference) Aşaması
inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
print(2)

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)
print(3)
# 5. Sonucu Yazdırma
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
print(4)
print("-" * 30)
print(response)
print("-" * 30)

In [ ]:
!pip install  faiss-cpu sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Daha güçlü multilingual model
embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")

def metni_parcalara_bol(metin, parca_boyutu=80, bindirme=20):
    """
    Daha küçük parçalar → daha hassas retrieval
    80 kelime: bir paragraf kadar, konuya odaklı
    """
    # Önce satırlara böl, boş satırları temizle
    satirlar = [s.strip() for s in metin.split("\n") if s.strip()]

    # Satırları birleştirerek parçala
    tam_metin = " ".join(satirlar)
    kelimeler = tam_metin.split()
    parcalar  = []

    i = 0
    while i < len(kelimeler):
        parca = kelimeler[i : i + parca_boyutu]
        parcalar.append(" ".join(parca))
        i += parca_boyutu - bindirme

    return [p for p in parcalar if len(p.strip()) > 20]  # Çok kısa parçaları at

def vektor_db_olustur(metin):
    parcalar = metni_parcalara_bol(metin)
    print(f"📄 Metin {len(parcalar)} parçaya bölündü.")

    # E5 modeli "query: " ve "passage: " prefix istiyor
    passage_parcalar = [f"passage: {p}" for p in parcalar]

    embeddings = embedding_model.encode(passage_parcalar, show_progress_bar=True, normalize_embeddings=True)
    embeddings = np.array(embeddings, dtype="float32")

    # L2 yerine Inner Product (normalize edilince cosine = inner product)
    boyut = embeddings.shape[1]
    index = faiss.IndexFlatIP(boyut)
    index.add(embeddings)

    print(f"✅ Vektör DB oluşturuldu! ({len(parcalar)} parça, {boyut} boyut)")
    return index, parcalar

def ilgili_parcalari_getir(soru, index, parcalar, top_k=4):
    # E5 modeli sorular için "query: " prefix istiyor
    soru_embedding = embedding_model.encode(
        [f"query: {soru}"],
        normalize_embeddings=True
    )
    soru_embedding = np.array(soru_embedding, dtype="float32")

    _, indeksler = index.search(soru_embedding, top_k)

    return [parcalar[idx] for idx in indeksler[0] if idx < len(parcalar)]

print("✅ RAG fonksiyonları hazır!")

In [ ]:
def prompt_olustur(kullanici_adi, soru, ilgili_bilgiler):
    baglam = "\n".join([f"- {b}" for b in ilgili_bilgiler])

    sistem_mesaji = f"""
Sen {kullanici_adi} adlı adayın kendisisin ve bir iş mülakatındasın.

Aşağıdaki KİŞİSEL BİLGİLER sana aittir ve cevap verirken bunlara sadık kalmalısın.
Cevabını birinci şahıs (ben dili) kullanarak ver.

Yanıtın:
- Doğal ve akıcı olsun (robotik olmasın)
- Gerçek bir mülakatta konuşuyormuş gibi profesyonel bir ton taşısın
- Gerektiğinde kısa örnekler veya deneyimlerle desteklensin
- Çok uzun olmasın (yaklaşık 4-6 cümle)

ÖNEMLİ:
- Bilgilerde olmayan hiçbir şeyi uydurma
- Sadece verilen bilgilere dayanarak konuş

KİŞİSEL BİLGİLER:
{baglam}
"""

    return [{
        "role": "user",
        "content": f"{sistem_mesaji}\n\nMülakat Sorusu: {soru}"
    }]

def cevap_uret(kullanici_adi, soru, index, parcalar, max_yeni_token=300):
    ilgili_bilgiler = ilgili_parcalari_getir(soru, index, parcalar, top_k=4)
    mesajlar = prompt_olustur(kullanici_adi, soru, ilgili_bilgiler)

    metin  = tokenizer.apply_chat_template(mesajlar, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(metin, return_tensors="pt").to(model.device)

    with torch.no_grad():
        cikti = model.generate(
            inputs["input_ids"],
            max_new_tokens=450,           # 500 → 300, HR cevabı kısa olmalı
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    yeni_tokenlar = cikti[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(yeni_tokenlar, skip_special_tokens=True), ilgili_bilgiler

print("✅ Cevap üretici hazır!")

In [ ]:
!pip install -q pymupdf

In [ ]:
import gradio as gr
from transformers import TextIteratorStreamer
from threading import Thread
import fitz

faiss_index     = None
metin_parcalari = None
KULLANICI_ADI   = ""

def pdf_oku(pdf_dosya):
    if pdf_dosya is None:
        return ""
    try:
        doc = fitz.open(pdf_dosya.name)
        metin = ""
        for sayfa in doc:
            metin += sayfa.get_text()
        doc.close()
        metin = metin.strip()
        if not metin:
            return "⚠️ PDF'den metin çıkarılamadı. Lütfen manuel girin."
        return metin
    except Exception as e:
        return f"⚠️ Hata: {str(e)}"

def profil_kaydet(kullanici_adi, kullanici_bilgisi):
    global faiss_index, metin_parcalari, KULLANICI_ADI
    if not kullanici_adi.strip() or not kullanici_bilgisi.strip():
        return (
            "⚠️ Lütfen isim ve bilgileri doldurun.",
            gr.update(visible=True),   # profil ekranı açık kalsın
            gr.update(visible=False),  # chat ekranı kapalı kalsın
        )
    KULLANICI_ADI = kullanici_adi.strip()
    faiss_index, metin_parcalari = vektor_db_olustur(kullanici_bilgisi)
    return (
        f"✅ Profil oluşturuldu!",
        gr.update(visible=False),  # profil ekranı kapat
        gr.update(visible=True),   # chat ekranı aç
    )

def soru_sor(soru, gecmis):
    if not soru.strip():
        yield gecmis, ""
        return

    gecmis = gecmis + [(soru, None)]
    yield gecmis, ""

    ilgili_bilgiler = ilgili_parcalari_getir(soru, faiss_index, metin_parcalari, top_k=3)
    mesajlar = prompt_olustur(KULLANICI_ADI, soru, ilgili_bilgiler)

    metin = tokenizer.apply_chat_template(mesajlar, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(metin, return_tensors="pt").to(model.device)

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    thread = Thread(target=model.generate, kwargs=dict(
        input_ids=inputs["input_ids"],
        max_new_tokens=500,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
        streamer=streamer,
    ))
    thread.start()

    birikim = ""
    for token in streamer:
        birikim += token
        yield gecmis[:-1] + [(soru, birikim)], ""

def profile_don(gecmis):
    """Profile dön butonuna basınca ekranı sıfırla."""
    return (
        gr.update(visible=True),   # profil ekranı aç
        gr.update(visible=False),  # chat ekranı kapat
        [],                         # sohbeti temizle
        [],
        "",                         # durum sıfırla
    )

with gr.Blocks(title="HR Mülakat Asistanı") as demo:

    # ════════════════════════════════
    # EKRAN 1 — Profil Oluştur
    # ════════════════════════════════
    with gr.Column(visible=True) as profil_ekrani:
        gr.Markdown("# HR Mülakat Asistanı")
        gr.Markdown("Bilgilerini gir veya CV yükle, profil oluştur ve mülakata başla.")

        isim_input = gr.Textbox(label="Adınız Soyadınız", placeholder="Ahmet Yılmaz")

        gr.Markdown("**CV yükle (PDF)** — metin alanına otomatik aktarılır")
        pdf_input  = gr.File(label="CV (PDF)", file_types=[".pdf"])
        pdf_btn    = gr.Button("PDF'den Aktar")

        bilgi_input = gr.Textbox(
            label="Bilgileriniz",
            placeholder="Deneyim, güçlü/zayıf yönler, hedefler...",
            lines=12
        )

        kaydet_btn = gr.Button("Mülakata Başla →", variant="primary")
        durum_text = gr.Textbox(label="Durum", interactive=False)

    # ════════════════════════════════
    # EKRAN 2 — Chatbot
    # ════════════════════════════════
    with gr.Column(visible=False) as chat_ekrani:
        with gr.Row():
            gr.Markdown("# Mülakat")
            geri_btn = gr.Button("← Profile Dön", scale=0)

        chatbot    = gr.Chatbot(label="", height=500)
        soru_input = gr.Textbox(label="HR Sorusu", placeholder="Soruyu yazın...")
        with gr.Row():
            gonder_btn  = gr.Button("Gönder", variant="primary")
            temizle_btn = gr.Button("Sohbeti Temizle")

    gecmis_state = gr.State([])

    # ── Eventler ──
    pdf_btn.click(fn=pdf_oku, inputs=pdf_input, outputs=bilgi_input)
    pdf_input.change(fn=pdf_oku, inputs=pdf_input, outputs=bilgi_input)

    kaydet_btn.click(
        fn=profil_kaydet,
        inputs=[isim_input, bilgi_input],
        outputs=[durum_text, profil_ekrani, chat_ekrani]
    )

    gonder_btn.click(
        fn=soru_sor,
        inputs=[soru_input, gecmis_state],
        outputs=[chatbot, soru_input]
    ).then(fn=lambda c: c, inputs=chatbot, outputs=gecmis_state)

    soru_input.submit(
        fn=soru_sor,
        inputs=[soru_input, gecmis_state],
        outputs=[chatbot, soru_input]
    ).then(fn=lambda c: c, inputs=chatbot, outputs=gecmis_state)

    temizle_btn.click(fn=lambda: ([], []), outputs=[chatbot, gecmis_state])

    geri_btn.click(
        fn=profile_don,
        inputs=gecmis_state,
        outputs=[profil_ekrani, chat_ekrani, chatbot, gecmis_state, durum_text]
    )

demo.launch(share=True, debug=True)

### RAG Performans Analizi


In [ ]:
!pip install rouge-score sentence-transformers

In [ ]:
import time
import numpy as np
from rouge_score import rouge_scorer
from sentence_transformers import util

# ── 1. RAG Retrieval Kalitesi ─────────────────────────────────────
def retrieval_skoru_hesapla(soru, getirilen_parcalar, kullanici_bilgisi):
    """
    Getirilen parçaların soruyla ne kadar alakalı olduğunu ölçer.
    Yöntem: Soru ve parçalar arasındaki cosine similarity (0-1 arası)
    """
    soru_embedding   = embedding_model.encode(soru, convert_to_tensor=True)
    parca_embeddingler = embedding_model.encode(getirilen_parcalar, convert_to_tensor=True)

    skorlar = util.cos_sim(soru_embedding, parca_embeddingler)[0]
    skorlar = skorlar.cpu().numpy()

    return {
        "en_yüksek": float(skorlar.max()),
        "ortalama" : float(skorlar.mean()),
        "en_düşük" : float(skorlar.min()),
        "tüm_skorlar": skorlar.tolist()
    }

# ── 2. Cevap Alakalılığı ─────────────────────────────────────────
def cevap_alakaliligi(soru, cevap):
    """
    Cevabın soruyla semantik benzerliğini ölçer.
    Düşükse model soruyu anlamamış olabilir.
    """
    soru_emb  = embedding_model.encode(soru, convert_to_tensor=True)
    cevap_emb = embedding_model.encode(cevap, convert_to_tensor=True)
    skor = util.cos_sim(soru_emb, cevap_emb).item()
    return round(skor, 4)



from sentence_transformers import util

def gercek_sadakat_olc(cevap, getirilen_parcalar):
    """
    ROUGE-L yerine semantik benzerlik ile sadakat ölçer.
    Cevap ile kaynak parçalar arasındaki max cosine similarity.
    """
    cevap_emb  = embedding_model.encode(f"passage: {cevap}", normalize_embeddings=True, convert_to_tensor=True)
    parca_embler = embedding_model.encode(
        [f"passage: {p}" for p in getirilen_parcalar],
        normalize_embeddings=True,
        convert_to_tensor=True
    )
    skorlar = util.cos_sim(cevap_emb, parca_embler)[0]
    return {
        "semantik_sadakat_max": round(float(skorlar.max()), 4),
        "semantik_sadakat_ort": round(float(skorlar.mean()), 4),
    }
# ── 3. Performans ────────────────────────────────────────────────
def performans_olc(kullanici_adi, soru, index, parcalar):
    """
    Uçtan uca süreyi ve token hızını ölçer.
    """
    # Retrieval süresi
    t0 = time.time()
    ilgili_bilgiler = ilgili_parcalari_getir(soru, index, parcalar, top_k=3)
    retrieval_suresi = time.time() - t0

    # Cevap üretim süresi
    mesajlar = prompt_olustur(kullanici_adi, soru, ilgili_bilgiler)
    metin    = tokenizer.apply_chat_template(mesajlar, tokenize=False, add_generation_prompt=True)
    inputs   = tokenizer(metin, return_tensors="pt").to(model.device)

    t1 = time.time()
    with torch.no_grad():
        cikti = model.generate(
            inputs["input_ids"],
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    uretim_suresi = time.time() - t1

    yeni_tokenlar = cikti[0][inputs["input_ids"].shape[-1]:]
    cevap         = tokenizer.decode(yeni_tokenlar, skip_special_tokens=True)
    token_sayisi  = len(yeni_tokenlar)

    return {
        "cevap"           : cevap,
        "ilgili_bilgiler" : ilgili_bilgiler,
        "retrieval_suresi": round(retrieval_suresi, 3),
        "uretim_suresi"   : round(uretim_suresi, 3),
        "toplam_sure"     : round(retrieval_suresi + uretim_suresi, 3),
        "token_sayisi"    : token_sayisi,
        "token_per_saniye": round(token_sayisi / uretim_suresi, 1),
    }

print("✅ Ölçüm fonksiyonları hazır!")

In [ ]:
# İstediğin soruları buraya ekle
test_sorulari = [
    "Kendinizden bahseder misiniz?",
    "En büyük güçlü ve zayıf yönleriniz nelerdir?",
    "Neden iş değiştirmek istiyorsunuz?",
    "5 yıl sonra kendinizi nerede görüyorsunuz?",
    "Takım yönetimi deneyiminiz var mı?",
]

sonuclar = []

print("🔍 Değerlendirme başlıyor...\n")
print("=" * 70)

for i, soru in enumerate(test_sorulari):
    print(f"\n[{i+1}/{len(test_sorulari)}] ❓ {soru}")
    print("-" * 70)

    # Performans + cevap üret
    perf = performans_olc(KULLANICI_ADI, soru, faiss_index, metin_parcalari)

    # Retrieval kalitesi
    retrieval = retrieval_skoru_hesapla(soru, perf["ilgili_bilgiler"], KULLANICI_BILGISI)

    # Cevap alakalılığı
    alakalilik = cevap_alakaliligi(soru, perf["cevap"])

    # Bilgiye sadakat
    sadakat2 = gercek_sadakat_olc(perf["cevap"], perf["ilgili_bilgiler"])

    # Kaydet
    sonuc = {
        "soru"              : soru,
        "cevap"             : perf["cevap"],
        "retrieval_ortalama": retrieval["ortalama"],
        "retrieval_max"     : retrieval["en_yüksek"],
        "cevap_alakaliligi" : alakalilik,
        "retrieval_suresi"  : perf["retrieval_suresi"],
        "uretim_suresi"     : perf["uretim_suresi"],
        "toplam_sure"       : perf["toplam_sure"],
        "token_sayisi"      : perf["token_sayisi"],
        "token_per_saniye"  : perf["token_per_saniye"],
    }
    sonuclar.append(sonuc)


    # Özet yazdır
    print(f"🤖 Cevap: {perf['cevap'][:200]}{'...' if len(perf['cevap']) > 200 else ''}")
    print(f"\n📊 Metrikler:")
    print(f"   Retrieval Benzerliği : {retrieval['ortalama']:.3f} (max: {retrieval['en_yüksek']:.3f})")
    print(f"   Cevap Alakalılığı    : {alakalilik:.3f}")
    print(f"   ⏱ Süre: retrieval={perf['retrieval_suresi']}s | üretim={perf['uretim_suresi']}s | toplam={perf['toplam_sure']}s")
    print(f"   🚀 Hız: {perf['token_sayisi']} token → {perf['token_per_saniye']} tok/sn")
    # Her soru için mevcut metriklerden sonra şunu ekle:

    print(f"   Semantik Sadakat (max): {sadakat2['semantik_sadakat_max']:.3f}")
    print(f"   Semantik Sadakat (ort): {sadakat2['semantik_sadakat_ort']:.3f}")

print("\n" + "=" * 70)
print("✅ Değerlendirme tamamlandı!")